# 01 — Interactive Model Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chris-L6/causalfm-survey/blob/main/notebooks/01_interactive_model_demo.ipynb)

**Choose a model and dataset, then run it!**

This notebook lets you interactively select from:
- **Models**: Foundation models (CausalPFN, Do-PFN, CausalFM) or traditional metalearners (S/T/X-learner, Debiased ML, IPW, DR)
- **Datasets**: Synthetic (linear, nonlinear, IV, frontdoor) or real-world (Lalonde)

All models follow a unified interface, so swapping between them is seamless.

## 1. Environment Setup

If running on **Colab**, this clones the repo. If **local**, it imports from parent directory.

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/chris-L6/causalfm-survey.git"
REPO_DIR = "causalfm-survey"

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    sys.path.insert(0, REPO_DIR)
else:
    sys.path.insert(0, os.path.abspath(".."))

import causal_bench
print("causal_bench imported from:", causal_bench.__file__)

In [ ]:
!pip install -q econml causalml causalpfn ipywidgets
import ipywidgets as widgets
import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

import torch
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Device: {device}")

## 2. Dataset Loader

In [ ]:
from causal_bench import get_dataset, load_lalonde, list_available_datasets

# Load all datasets
DATASETS = {}
print("Loading synthetic datasets...")
for name in ["linear_confounded", "nonlinear_heterogeneous", "iv_binary", "frontdoor"]:
    DATASETS[name] = get_dataset(name, n=2000, seed=0)

print("Loading Lalonde dataset...")
try:
    DATASETS["lalonde_nsw_psid"] = load_lalonde()
    print("  ✓ Lalonde loaded")
except Exception as e:
    print(f"  ✗ Lalonde unavailable: {e}")

print(f"\nAvailable datasets: {list(DATASETS.keys())}")
for name, ds in DATASETS.items():
    print(f"  {name}: n={len(ds.Y)}, X.shape={ds.X.shape}, ATE={ds.ate:.3f}")

## 3. Model Selection Widgets

In [ ]:
from causal_bench import (
    CausalPFNWrapper, DoPFNWrapper, CausalFMWrapper,
    SLearnerWrapper, TLearnerWrapper, XLearnerWrapper,
    DebiasedMLWrapper, IPWWrapper, DRWrapper
)

MODELS = {
    "CausalPFN": CausalPFNWrapper,
    "Do-PFN": DoPFNWrapper,
    "CausalFM": CausalFMWrapper,
    "S-learner": SLearnerWrapper,
    "T-learner": TLearnerWrapper,
    "X-learner": XLearnerWrapper,
    "Debiased ML": DebiasedMLWrapper,
    "IPW": IPWWrapper,
    "DR (Doubly Robust)": DRWrapper,
}

# Check availability
available_models = {name: cls for name, cls in MODELS.items() if cls.is_available()}
print(f"Available models ({len(available_models)}/{len(MODELS)}):")
for name in available_models:
    print(f"  ✓ {name}")
for name in set(MODELS.keys()) - set(available_models.keys()):
    print(f"  ✗ {name}")

# Widgets for selection
model_dropdown = widgets.Dropdown(options=available_models.keys(), description="Model:")
dataset_dropdown = widgets.Dropdown(options=DATASETS.keys(), description="Dataset:")

display(widgets.VBox([model_dropdown, dataset_dropdown]))

## 4. Run Selected Model on Selected Dataset

In [ ]:
from causal_bench import evaluate_cate

def run_demo():
    model_name = model_dropdown.value
    dataset_name = dataset_dropdown.value

    print(f"\n{'='*60}")
    print(f"Running {model_name} on {dataset_name}")
    print(f"{'='*60}\n")

    ds = DATASETS[dataset_name]
    train_idx, test_idx = ds.train_test_split(0.7, seed=0)

    X_train, X_test = ds.X[train_idx], ds.X[test_idx]
    T_train, Y_train = ds.T[train_idx], ds.Y[train_idx]

    # For synthetic datasets, we have ground truth tau
    if hasattr(ds, 'tau'):
        tau_test = ds.tau[test_idx]
    else:
        tau_test = None

    try:
        t0 = time.time()
        model_cls = MODELS[model_name]

        # Special handling for CausalFM (needs checkpoint path)
        if model_name == "CausalFM":
            checkpoint_path = "CausalFM-toolkit/checkpoints/best_model.pth"
            if not os.path.exists(checkpoint_path):
                raise FileNotFoundError(f"CausalFM checkpoint not found at {checkpoint_path}")
            model = model_cls(checkpoint_path=checkpoint_path, device=device)
        else:
            model = model_cls(device=device) if "Wrapper" in model_cls.__name__ and "Foundation" not in model_name else model_cls()

        model.fit(X_train, T_train, Y_train)
        tau_hat, lower, upper = model.predict(X_test)
        runtime = time.time() - t0

        ate_hat = float(np.mean(tau_hat))

        if tau_test is not None:
            result = evaluate_cate(tau_hat, tau_test, ate_hat=ate_hat, ate_true=ds.ate,
                                   lower=lower, upper=upper, runtime_s=runtime)
            print("Results:")
            for key, val in result.items():
                if val is not None:
                    print(f"  {key:20s}: {val:10.4f}")
        else:
            print(f"  ATE_hat:          {ate_hat:.4f}")
            print(f"  ATE_true (observed): {ds.ate:.4f}")
            print(f"  Runtime:          {runtime:.2f}s")
            print("  (Ground truth CATE unavailable for real data)")

    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()

# Create run button
run_button = widgets.Button(description="▶ Run Demo")
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output(wait=False)
        run_demo()

run_button.on_click(on_click)

display(widgets.VBox([run_button, output]))

## Notes

- **Foundation models** (CausalPFN, Do-PFN, CausalFM) use in-context learning: they don't re-train on your data, just condition on it.
- **Metalearners** (S/T/X, Debiased ML, etc.) are traditional ML methods trained separately for control and treatment.
- **Synthetic datasets** have ground-truth CATE (`tau`), so full metrics (PEHE, bias, coverage) are computed.
- **Lalonde** is real-world data: no ground truth CATE, only observed ATE is available.
- **Missing models** (shown with ✗) require additional installs; check the notebook setup cells.